In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
%pip install -q dagshub mlflow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 685.1 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 3.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 29.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━

In [3]:
import dagshub
dagshub.init(repo_owner='mesata', repo_name='IEE-CIS-Fraud-Detection', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=2c3ec139-016d-4e4b-a85f-7759b678d862&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=479348eeb0355fe8c1a1f1bbfc0aea023e8c175b07ffcfbda156f1c7f8201684




Output()

Accessing as mesata

Initialized MLflow to track repo "mesata/IEE-CIS-Fraud-Detection"

Repository mesata/IEE-CIS-Fraud-Detection initialized!

🏃 View run gentle-shoat-169 at: https://dagshub.com/mesata/IEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/f20a021c3b534cbfb7bd5eb085469d7a
🧪 View experiment at: https://dagshub.com/mesata/IEE-CIS-Fraud-Detection.mlflow/#/experiments/0


# CLEANING


In [10]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

In [11]:
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
del train_transaction, train_identity

In [13]:
nan_stats = train.isnull().mean()
cols_to_drop = nan_stats[nan_stats > 0.8].index
train = train.drop(columns=cols_to_drop) 
num_cols = train.select_dtypes(include=[np.number]).columns
train[num_cols] = train[num_cols].fillna(train[num_cols].median())
print(f"წაიშალა {len(cols_to_drop)} სვეტი. დარჩა {train.shape[1]} სვეტი.")

წაიშალა 69 სვეტი. დარჩა 365 სვეტი.


# FEATURE ENGINEERING

In [15]:
train['TransactionHour'] = (train['TransactionDT'] // 3600) % 24


In [12]:
from sklearn.preprocessing import LabelEncoder
cat_cols = train.select_dtypes(include=['object']).columns
for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))

print(f"Encrypted {len(cat_cols)} categorical columns.")

Encrypted 31 categorical columns.


# FEATURE SELECTION

In [16]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01) 
train_selected = train.copy()
num_features = train.select_dtypes(include=[np.number]).columns
selector.fit(train[num_features])
features_to_keep = num_features[selector.get_support()]
all_features = list(features_to_keep) + list(train.select_dtypes(include=['object']).columns)
train = train[all_features]
print(f"Variance Threshold-ის შემდეგ დარჩა {train.shape[1]} სვეტი.")
corr_matrix = train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
train = train.drop(columns=to_drop)
print(f"მაღალი კორელაციის გამო წაიშალა {len(to_drop)} სვეტი.")
print(f"საბოლოო ჯამში დაგვრჩა {train.shape[1]} ყველაზე ინფორმატიული სვეტი.")

Variance Threshold-ის შემდეგ დარჩა 342 სვეტი.
მაღალი კორელაციის გამო წაიშალა 90 სვეტი.
საბოლოო ჯამში დაგვრჩა 252 ყველაზე ინფორმატიული სვეტი.


# TRAINING

In [18]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
X = train.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1, errors='ignore')
y = train['isFraud']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
with mlflow.start_run(run_name="AdaBoost_Final_Experiment"):
    ada_model = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
    print("Training AdaBoost")
    ada_model.fit(X_train, y_train)
    y_pred_proba = ada_model.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, y_pred_proba)
    mlflow.log_param("model_type", "AdaBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("auc", auc_score)
    print(f"Validation AUC Score: {auc_score:.4f}")

Training AdaBoost
Validation AUC Score: 0.8374
🏃 View run AdaBoost_Final_Experiment at: https://dagshub.com/mesata/IEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/ae172f33a8b244e996f0ced33f64888a
🧪 View experiment at: https://dagshub.com/mesata/IEE-CIS-Fraud-Detection.mlflow/#/experiments/0
